In [262]:
import os
import gzip
import pandas as pd
import matplotlib.pyplot as plt
import re
import numpy as np
from google.cloud import bigquery

In [263]:
client = bigquery.Client()
dataset_id = "physionet-data"

**ICU Data Pre-Check**

In [264]:
query = f"""
SELECT *
FROM `{dataset_id}.mimiciv_3_1_icu.icustays`
"""
df_icu_stay = client.query(query).to_dataframe()
df_icu_stay.head()

,subject_id,hadm_id,stay_id,first_careunit,last_careunit,intime,outtime,los
0,10270110,20171261,35854639,PACU,PACU,2134-03-25 03:32:02,2134-03-25 14:20:42,0.450463
1,10270110,20171261,36372959,PACU,PACU,2134-03-24 01:31:39,2134-03-25 03:31:52,1.083484
2,10270644,20019675,35548343,PACU,PACU,2159-12-03 16:20:31,2159-12-08 17:28:42,5.047350
3,10368426,21588639,39194905,PACU,PACU,2164-12-30 13:29:21,2164-12-30 14:00:38,0.021725
4,10464753,28216499,32421516,PACU,PACU,2183-01-10 20:51:04,2183-01-11 22:58:45,1.088669


In [265]:
len(df_icu_stay['stay_id'].unique())

94458

In [266]:
len(df_icu_stay['subject_id'].unique())

65366

# Component A

In [267]:
regex= r"\b(?:cpot[-\s:]*)?muscle\s+tension(?:\s*\(cpot[ab]\))?\b"
regex_output= client.query(f"""
SELECT *
FROM `{dataset_id}.mimiciv_3_1_icu.d_items`
WHERE REGEXP_CONTAINS(
  LOWER(CONCAT(' ',
    IFNULL(label,''),' ',
    IFNULL(abbreviation,''),' '
  )),
  r\"\"\"{regex}\"\"\"
)
""").to_dataframe()
regex_output

,itemid,label,abbreviation,linksto,category,unitname,param_type,lownormalvalue,highnormalvalue
0,229691,CPOT-Muscle Tension (CPOTb),Muscle Tension,chartevents,Pain/Sedation,None,Text,NaN,NaN
1,229692,CPOT-Muscle Tension (CPOTa),Muscle Tension-Post,chartevents,Pain/Sedation,None,Text,NaN,NaN


In [268]:
regex= r"\b(?:cpot[-\s:]*)?vocali[sz]ation(?:\s*\(cpot[ab]\))?(?:_old_\d+)?\b"
regex_output= client.query(f"""
SELECT *
FROM `{dataset_id}.mimiciv_3_1_icu.d_items`
WHERE REGEXP_CONTAINS(
  LOWER(CONCAT(' ',
    IFNULL(label,''),' ',
    IFNULL(abbreviation,''),' '
  )),
  r\"\"\"{regex}\"\"\"
)
""").to_dataframe()
regex_output

,itemid,label,abbreviation,linksto,category,unitname,param_type,lownormalvalue,highnormalvalue
0,229693,Vocalization (CPOTb)_OLD_1,Vocalization (CPOTb)_OLD_1,chartevents,Pain/Sedation,None,Text,NaN,NaN
1,229694,CPOT-Vocalization (CPOTa),Vocalization-Post,chartevents,Pain/Sedation,None,Text,NaN,NaN
2,229695,CPOT-Vocalization (CPOTb),Vocalization,chartevents,Pain/Sedation,None,Text,NaN,NaN


In [269]:
regex= r"\b(?:cpot[-\s:]*)?body\s+movements?(?:\s*\(cpot[ab]\))?\b"
regex_output= client.query(f"""
SELECT *
FROM `{dataset_id}.mimiciv_3_1_icu.d_items`
WHERE REGEXP_CONTAINS(
  LOWER(CONCAT(' ',
    IFNULL(label,''),' ',
    IFNULL(abbreviation,''),' '
  )),
  r\"\"\"{regex}\"\"\"
)
""").to_dataframe()
regex_output

,itemid,label,abbreviation,linksto,category,unitname,param_type,lownormalvalue,highnormalvalue
0,229696,CPOT-Body Movements (CPOTb),Body Movements,chartevents,Pain/Sedation,None,Text,NaN,NaN
1,229697,CPOT-Body Movements (CPOTa),Body Movements-Post,chartevents,Pain/Sedation,None,Text,NaN,NaN


In [270]:
regex= r"\b(?:cpot[-\s:]*)?facial\s+expression(?:\s*\(cpot[ab]\))?\b"
regex_output= client.query(f"""
SELECT *
FROM `{dataset_id}.mimiciv_3_1_icu.d_items`
WHERE REGEXP_CONTAINS(
  LOWER(CONCAT(' ',
    IFNULL(label,''),' ',
    IFNULL(abbreviation,''),' '
  )),
  r\"\"\"{regex}\"\"\"
)
""").to_dataframe()
regex_output

,itemid,label,abbreviation,linksto,category,unitname,param_type,lownormalvalue,highnormalvalue
0,229698,CPOT-Facial Expression (CPOTb),Facial Expression,chartevents,Pain/Sedation,None,Text,NaN,NaN
1,229699,CPOT-Facial Expression (CPOTa),Facial Expression-Post,chartevents,Pain/Sedation,None,Text,NaN,NaN


**Alternative Way to Extract Concepts**

In [271]:
cpot_d_items_filtered = client.query(f'''
SELECT * FROM `{dataset_id}.mimiciv_3_1_icu.d_items`
WHERE label LIKE '%%CPOT%%'
OR abbreviation LIKE '%%CPOT%%'
''').to_dataframe()
cpot_d_items_filtered

,itemid,label,abbreviation,linksto,category,unitname,param_type,lownormalvalue,highnormalvalue
0,229689,CPOT-Pain Assessment Method,CPOT-Pain Assessment Method,chartevents,Pain/Sedation,None,Text,NaN,NaN
1,229690,CPOT-Pain Assessment Method-Post,Pain Assessment Method-Post,chartevents,Pain/Sedation,None,Text,NaN,NaN
2,229691,CPOT-Muscle Tension (CPOTb),Muscle Tension,chartevents,Pain/Sedation,None,Text,NaN,NaN
3,229692,CPOT-Muscle Tension (CPOTa),Muscle Tension-Post,chartevents,Pain/Sedation,None,Text,NaN,NaN
4,229693,Vocalization (CPOTb)_OLD_1,Vocalization (CPOTb)_OLD_1,chartevents,Pain/Sedation,None,Text,NaN,NaN
5,229694,CPOT-Vocalization (CPOTa),Vocalization-Post,chartevents,Pain/Sedation,None,Text,NaN,NaN
6,229695,CPOT-Vocalization (CPOTb),Vocalization,chartevents,Pain/Sedation,None,Text,NaN,NaN
7,229696,CPOT-Body Movements (CPOTb),Body Movements,chartevents,Pain/Sedation,None,Text,NaN,NaN
8,229697,CPOT-Body Movements (CPOTa),Body Movements-Post,chartevents,Pain/Sedation,None,Text,NaN,NaN
9,229698,CPOT-Facial Expression (CPOTb),Facial Expression,chartevents,Pain/Sedation,None,Text,NaN,NaN


In [272]:
cpot_related_itemids = [
    229689,  # CPOT-Pain Assessment Method
    229690,  # CPOT-Pain Assessment Method-Post
    229691,  # CPOT-Muscle Tension (CPOTb)
    229692,  # CPOT-Muscle Tension (CPOTa)
    229694,  # CPOT-Vocalization (CPOTa)
    229695,  # CPOT-Vocalization (CPOTb)
    229696,  # CPOT-Body Movements (CPOTb)
    229697,  # CPOT-Body Movements (CPOTa)
    229698,  # CPOT-Facial Expression (CPOTb)
    229699,  # CPOT-Facial Expression (CPOTa)
    229703   # CPOT-Pain Management
]

In [273]:
query = f"""
SELECT *
FROM `{dataset_id}.mimiciv_3_1_icu.chartevents`
WHERE itemid IN ({', '.join(map(str, cpot_related_itemids))})
"""

In [274]:
df_cpot = client.query(query).to_dataframe()
df_cpot.head()

,subject_id,hadm_id,stay_id,caregiver_id,charttime,storetime,itemid,value,valuenum,valueuom,warning
0,16713311,21147739,37117646,84589,2170-03-23 08:00:00,2170-03-23 09:22:00,229691,Very tense or rigid,2.0,None,0
1,19846331,24497785,36635669,49727,2177-09-19 08:00:00,2177-09-19 10:20:00,229691,Very tense or rigid,2.0,None,0
2,10525750,20025423,35914661,35113,2110-01-17 18:03:00,2110-01-17 20:03:00,229695,"Crying out, sobbing OR Fighting vent/asynchron...",2.0,None,0
3,11810038,21902679,32353102,17994,2129-08-22 12:50:00,2129-08-22 16:08:00,229695,"Crying out, sobbing OR Fighting vent/asynchron...",2.0,None,0
4,17477734,24766196,34564006,60594,2117-02-13 08:00:00,2117-02-13 07:47:00,229703,Epidural,NaN,None,0


In [275]:
df_cpot.shape

(2122133, 11)

In [276]:
cpot_related_itemids = {
    229691: 'CPOT-Muscle Tension (CPOTb)',
    229692: 'CPOT-Muscle Tension (CPOTa)',
    229694: 'CPOT-Vocalization (CPOTa)',
    229695: 'CPOT-Vocalization (CPOTb)',
    229696: 'CPOT-Body Movements (CPOTb)',
    229697: 'CPOT-Body Movements (CPOTa)',
    229698: 'CPOT-Facial Expression (CPOTb)',
    229699: 'CPOT-Facial Expression (CPOTa)',
    }
cpot_dict = {}

# Iterate over each itemid in the prioritized list with descriptions
for itemid, description in cpot_related_itemids.items():
    # Filter the DataFrame for rows with the current itemid
    filtered_df = df_cpot[df_cpot['itemid'] == itemid]

    # Get the unique values for these rows and their value counts
    unique_values = filtered_df['value'].value_counts().to_dict()

    # Assign a dictionary with description, unique values, and their counts to the itemid in the final dictionary
    cpot_dict[itemid] = {"data description": description, "data unique values": unique_values}

cpot_dict

{229691: {'data description': 'CPOT-Muscle Tension (CPOTb)',
  'data unique values': {'Relaxed': 206492,
   'Tense, rigid': 41812,
   'Very tense or rigid': 6269}},
 229692: {'data description': 'CPOT-Muscle Tension (CPOTa)',
  'data unique values': {'Relaxed': 78617,
   'Tense, rigid': 5848,
   'Very tense or rigid': 522}},
 229694: {'data description': 'CPOT-Vocalization (CPOTa)',
  'data unique values': {'Talking in normal tone/no sound OR Tolerating ventilator/No alarms': 81373,
   'Sighing, moaning OR Intermittent vent alarms, coughing, but tolerating vent': 3601,
   'Crying out, sobbing OR Fighting vent/asynchrony, frequent alarms': 316}},
 229695: {'data description': 'CPOT-Vocalization (CPOTb)',
  'data unique values': {'Talking in normal tone/no sound OR Tolerating ventilator/No alarms': 219417,
   'Sighing, moaning OR Intermittent vent alarms, coughing, but tolerating vent': 29081,
   'Crying out, sobbing OR Fighting vent/asynchrony, frequent alarms': 6062}},
 229696: {'data 

In [277]:
relevant_item_ids = [229691, 229692, 229694, 229695, 229696, 229697, 229698, 229699]
df_cpot = df_cpot[df_cpot['itemid'].isin(relevant_item_ids)]

In [278]:
df_cpot['subject_id'].nunique()

11165

In [279]:
muscle_tension_mapping = {'Relaxed': 0, 'Tense, rigid': 1, 'Very tense or rigid': 2}
vocalization_mapping = {
    'Talking in normal tone/no sound OR Tolerating ventilator/No alarms': 0,
    'Sighing, moaning OR Intermittent vent alarms, coughing, but tolerating vent': 1,
    'Crying out, sobbing OR Fighting vent/asynchrony, frequent alarms': 2
}
body_movements_mapping = {'Absence of movements, or normal position': 0, 'Protection/guarding/withdraws': 1, 'Restlessness/thrashing': 2}
facial_expression_mapping = {'Relaxed, Neutral': 0, 'Tense': 1, 'Grimacing': 2}

In [280]:
df_cpot['muscle_tension_score'] = df_cpot.apply(
    lambda row: muscle_tension_mapping.get(row['value'], 0) if row['itemid'] in [229691, 229692] else 0, axis=1)
df_cpot['vocalization_score'] = df_cpot.apply(
    lambda row: vocalization_mapping.get(row['value'], 0) if row['itemid'] in [229695, 229694] else 0, axis=1)
df_cpot['body_movements_score'] = df_cpot.apply(
    lambda row: body_movements_mapping.get(row['value'], 0) if row['itemid'] in [229696, 229697] else 0, axis=1)
df_cpot['facial_expression_score'] = df_cpot.apply(
    lambda row: facial_expression_mapping.get(row['value'], 0) if row['itemid'] in [229698, 229699] else 0, axis=1)

In [281]:
df_cpot['charttime'] = pd.to_datetime(df_cpot['charttime'])

df_cpot.set_index('charttime', inplace=True)

df_cpot['date'] = df_cpot.index.date
daily_scores = df_cpot.groupby(['subject_id', 'date']).agg({
    'muscle_tension_score': 'sum',
    'vocalization_score': 'sum',
    'body_movements_score': 'sum',
    'facial_expression_score': 'sum'
})

daily_scores['total_cpot_score'] = daily_scores.sum(axis=1)

In [282]:
daily_scores.info()

<class 'pandas.core.frame.DataFrame'>
MultiIndex: 60549 entries, (np.int64(10002114), datetime.date(2162, 2, 18)) to (np.int64(19997886), datetime.date(2186, 12, 7))
Data columns (total 5 columns):
 #   Column                   Non-Null Count  Dtype
---  ------                   --------------  -----
 0   muscle_tension_score     60549 non-null  int64
 1   vocalization_score       60549 non-null  int64
 2   body_movements_score     60549 non-null  int64
 3   facial_expression_score  60549 non-null  int64
 4   total_cpot_score         60549 non-null  int64
dtypes: int64(5)
memory usage: 2.8 MB


In [283]:
daily_scores.head()

muscle_tension_score  vocalization_score  \
subject_id date                                                   
10002114   2162-02-18                     2                   2   
           2162-02-19                     0                   0   
10002667   2187-02-24                     1                   0   
10003637   2150-05-16                     0                   0   
           2150-05-17                     1                   0   

                       body_movements_score  facial_expression_score  \
subject_id date                                                        
10002114   2162-02-18                     3                        4   
           2162-02-19                     0                        0   
10002667   2187-02-24                     1                        3   
10003637   2150-05-16                     0                        0   
           2150-05-17                     1                        1   

                       total_cpot_score  
subject_id date                          
10002114   2162-02-18                11  
           2162-02-19                 0  
10002667   2187-02-24                 5  
10003637   2150-05-16                 0  
           2150-05-17                 3

In [284]:
daily_scores.index.get_level_values('subject_id').nunique()

11165

In [285]:
pain_assessments_per_day  = df_cpot.groupby(['subject_id', 'date']).size().reset_index(name='counts')

# Significant pain assessments (CPOT > 2)
significant_pain = daily_scores[daily_scores['total_cpot_score'] > 2].groupby(['subject_id', 'date']).size()

pain_assessments_compliance = pain_assessments_per_day[pain_assessments_per_day['counts'] > 6]

compliance = (pain_assessments_per_day[pain_assessments_per_day['counts'] >= 6].count() / pain_assessments_per_day.count()) * 100
high_scores = daily_scores[daily_scores['total_cpot_score'] > 2]

In [286]:
pain_assessments_per_day.head(5)

,subject_id,date,counts
0,10002114,2162-02-18,28
1,10002114,2162-02-19,8
2,10002667,2187-02-24,12
3,10003637,2150-05-16,28
4,10003637,2150-05-17,36


Total Paitent Days

In [287]:
len(pain_assessments_per_day)

60549

In [288]:
# Assessments per patient per day
assessments_per_day = df_cpot.groupby(['subject_id', 'date']).size().reset_index(name='counts')

# Compliant days (where assessments >= 6)
compliant_days = assessments_per_day[assessments_per_day['counts'] >= 6]

# Compliance percentage
compliance_rate = (compliant_days.shape[0] / assessments_per_day.shape[0]) * 100
median_assessments = assessments_per_day['counts'].median()
iqr_assessments = assessments_per_day['counts'].quantile(0.75) - assessments_per_day['counts'].quantile(0.25)
mean_assessments = assessments_per_day['counts'].mean()
std_assessments = assessments_per_day['counts'].std()

# Minimum and maximum assessments per day
min_assessments = assessments_per_day['counts'].min()
max_assessments = assessments_per_day['counts'].max()

print("Compliance Rate (%):", compliance_rate)
print("Median assessments per day:", median_assessments)
print("Interquartile range (IQR):", iqr_assessments)
print("Mean assessments per day:", mean_assessments)
print("Standard deviation (SD):", std_assessments)
print("Minimum assessments per day:", min_assessments)
print("Maximum assessments per day:", max_assessments)

Compliance Rate (%): 88.76116864027482
Median assessments per day: 23.0
Interquartile range (IQR): 20.0
Mean assessments per day: 22.503790318584947
Standard deviation (SD): 13.743818768415759
Minimum assessments per day: 1
Maximum assessments per day: 124


In [289]:
compliant_days = pain_assessments_per_day[pain_assessments_per_day['counts'] >= 6]

# Compliance percentage
compliance_rate = (compliant_days.count() / pain_assessments_per_day.count()) * 100

# Median and IQR for pain assessments per day
median_assessments = pain_assessments_per_day['counts'].median()
iqr_assessments = pain_assessments_per_day['counts'].quantile(0.75) - pain_assessments_per_day['counts'].quantile(0.25)

print("Compliance Rate (%):", compliance_rate)
print("Median assessments per day:", median_assessments)
print("Interquartile range (IQR):", iqr_assessments)

Compliance Rate (%): subject_id    88.761169
date          88.761169
counts        88.761169
dtype: float64
Median assessments per day: 23.0
Interquartile range (IQR): 20.0


In [290]:
pain_assessments_per_day['counts'].describe()

,counts
count,60549.000000
mean,22.503790
std,13.743819
min,1.000000
25%,12.000000
50%,23.000000
75%,32.000000
max,124.000000


In [291]:
summary = pain_assessments_per_day['counts'].describe()
print(summary)

count    60549.000000
mean        22.503790
std         13.743819
min          1.000000
25%         12.000000
50%         23.000000
75%         32.000000
max        124.000000
Name: counts, dtype: float64


In [292]:
daily_scores.index.get_level_values('subject_id').nunique()

11165

In [293]:
pain_assessments_compliance

,subject_id,date,counts
0,10002114,2162-02-18,28
1,10002114,2162-02-19,8
2,10002667,2187-02-24,12
3,10003637,2150-05-16,28
4,10003637,2150-05-17,36
...,...,...,...
60544,19997843,2120-11-18,9
60545,19997843,2120-11-19,16
60546,19997843,2120-11-20,20
60547,19997886,2186-12-06,20


In [294]:
significant_pain.index.get_level_values('subject_id').nunique()

6914

In [295]:
significant_pain.index.nunique()

24320

# Component B

In [296]:
regex= r"\bventilator[-\s:]*mode\b"
regex_output= client.query(f"""
SELECT *
FROM `{dataset_id}.mimiciv_3_1_icu.d_items`
WHERE REGEXP_CONTAINS(
  LOWER(CONCAT(' ',
    IFNULL(label,''),' ',
    IFNULL(abbreviation,''),' '
  )),
  r\"\"\"{regex}\"\"\"
)
""").to_dataframe()
regex_output

,itemid,label,abbreviation,linksto,category,unitname,param_type,lownormalvalue,highnormalvalue
0,223849,Ventilator Mode,Ventilator Mode,chartevents,Respiratory,None,Text,NaN,NaN
1,229314,Ventilator Mode (Hamilton),Ventilator Mode (Hamilton),chartevents,Respiratory,None,Text,NaN,NaN


In [297]:
regex= r"\bsbt[-\s:]?started?\b"
regex_output= client.query(f"""
SELECT *
FROM `{dataset_id}.mimiciv_3_1_icu.d_items`
WHERE REGEXP_CONTAINS(
  LOWER(CONCAT(' ',
    IFNULL(label,''),' ',
    IFNULL(abbreviation,''),' '
  )),
  r\"\"\"{regex}\"\"\"
)
""").to_dataframe()
regex_output

,itemid,label,abbreviation,linksto,category,unitname,param_type,lownormalvalue,highnormalvalue
0,224715,SBT Started,SBT Started,chartevents,Respiratory,None,Text,NaN,NaN


In [298]:
regex= r"\bsbt[-\s:]?success(?:fully)?\b"
regex_output= client.query(f"""
SELECT *
FROM `{dataset_id}.mimiciv_3_1_icu.d_items`
WHERE REGEXP_CONTAINS(
  LOWER(CONCAT(' ',
    IFNULL(label,''),' ',
    IFNULL(abbreviation,''),' '
  )),
  r\"\"\"{regex}\"\"\"
)
""").to_dataframe()
regex_output

,itemid,label,abbreviation,linksto,category,unitname,param_type,lownormalvalue,highnormalvalue
0,224717,SBT Successfully Completed,SBT Successfully Completed,chartevents,Respiratory,None,Text,NaN,NaN


In [299]:
regex= r"\bsbt[-\s:]?stopped?\b"
regex_output= client.query(f"""
SELECT *
FROM `{dataset_id}.mimiciv_3_1_icu.d_items`
WHERE REGEXP_CONTAINS(
  LOWER(CONCAT(' ',
    IFNULL(label,''),' ',
    IFNULL(abbreviation,''),' '
  )),
  r\"\"\"{regex}\"\"\"
)
""").to_dataframe()
regex_output

,itemid,label,abbreviation,linksto,category,unitname,param_type,lownormalvalue,highnormalvalue
0,224716,SBT Stopped,SBT Stopped,chartevents,Respiratory,None,Text,NaN,NaN


In [300]:
regex= r"\bsbt[-\s:]?defer(?:red)?\b"
regex_output= client.query(f"""
SELECT *
FROM `{dataset_id}.mimiciv_3_1_icu.d_items`
WHERE REGEXP_CONTAINS(
  LOWER(CONCAT(' ',
    IFNULL(label,''),' ',
    IFNULL(abbreviation,''),' '
  )),
  r\"\"\"{regex}\"\"\"
)
""").to_dataframe()
regex_output

,itemid,label,abbreviation,linksto,category,unitname,param_type,lownormalvalue,highnormalvalue
0,224833,SBT Deferred,SBT Deferred,chartevents,Respiratory,None,Text,NaN,NaN


**Alternative Way to Extract Concepts**

In [301]:
ventilation_d_items_filtered = client.query(f'''
SELECT * FROM `{dataset_id}.mimiciv_3_1_icu.d_items`
WHERE label LIKE '%%Ventilator%%'
OR abbreviation LIKE '%%Ventilation%%'
''').to_dataframe()
ventilation_d_items_filtered

,itemid,label,abbreviation,linksto,category,unitname,param_type,lownormalvalue,highnormalvalue
0,225792,Invasive Ventilation,Invasive Ventilation,procedureevents,2-Ventilation,None,Processes,NaN,NaN
1,225794,Non-invasive Ventilation,Non-invasive Ventilation,procedureevents,2-Ventilation,None,Processes,NaN,NaN
2,225303,Mask Ventilation (Intubation),Mask Ventilation (Intubation),chartevents,Intubation,None,Text,NaN,NaN
3,223848,Ventilator Type,Ventilator Type,chartevents,Respiratory,None,Text,NaN,NaN
4,223849,Ventilator Mode,Ventilator Mode,chartevents,Respiratory,None,Text,NaN,NaN
5,229314,Ventilator Mode (Hamilton),Ventilator Mode (Hamilton),chartevents,Respiratory,None,Text,NaN,NaN
6,227565,Ventilator Tank #1,Ventilator Tank #1,chartevents,Respiratory,None,Numeric,NaN,NaN
7,227566,Ventilator Tank #2,Ventilator Tank #2,chartevents,Respiratory,None,Numeric,NaN,NaN


In [302]:
ventilation_related_itemids = [
    223848,  # Ventilator Type
    223849,  # Ventilator Mode
    225792,  # Invasive Ventilation
    229314,  # Ventilator Mode (Hamilton)
    224715,  # SBT Started
    224717,  # SBT Successfully Completed
    224716,  # SBT Stopped
    224833,  # SBT Deferred
]

In [303]:
query = f"""
SELECT *
FROM `{dataset_id}.mimiciv_3_1_icu.chartevents`
WHERE itemid IN ({', '.join(map(str, ventilation_related_itemids))})
"""

In [304]:
df_ventilation = client.query(query).to_dataframe()
df_ventilation.head()

,subject_id,hadm_id,stay_id,caregiver_id,charttime,storetime,itemid,value,valuenum,valueuom,warning
0,18566230,23391766,36789206,45039,2166-05-02 04:59:00,2166-05-02 04:59:00,224833,Pending procedure,NaN,None,0
1,10074387,25872015,36576905,76295,2142-12-01 02:00:00,2142-12-01 02:33:00,224833,Pending procedure,NaN,None,1
2,19940147,29892428,32144801,95841,2127-11-17 04:00:00,2127-11-17 04:34:00,224833,Pending procedure,NaN,None,0
3,10336855,24910923,33336719,39388,2151-07-09 04:00:00,2151-07-09 04:07:00,224833,Pending procedure,NaN,None,0
4,12880983,29028734,34337476,31733,2125-02-20 04:00:00,2125-02-20 03:28:00,224833,Pending procedure,NaN,None,0


In [305]:
df_ventilation['itemid'].unique()

<IntegerArray>
[224833, 223848, 223849, 229314, 224716, 224715, 224717]
Length: 7, dtype: Int64

In [306]:
vmodes=[223849,229314]
df_Ventilator=df_ventilation[df_ventilation['itemid'].isin(vmodes)]
df_Ventilator[df_Ventilator['itemid'].isin(vmodes)]['value'].unique()

array(['CMV', 'APRV', 'CPAP', 'VOL/AC', 'PRVC/AC', 'PCV+/PSV',
       'PCV+Assist', 'CPAP/PSV+ApnVol', 'SIMV/PSV/AutoFlow', 'P-CMV',
       'MMV/PSV', 'SIMV/PSV', 'CMV/AutoFlow', 'CPAP/PSV+ApnPres', 'VS',
       'NIV', 'Ambient', 'PCV+', 'CPAP/PPS', '(S) CMV', 'MMV', 'PRES/AC',
       'APRV/Biphasic+ApnVol', 'MMV/AutoFlow', 'SIMV/PRES', 'APV (simv)',
       'Apnea Ventilation', 'SIMV', 'SYNCHRON MASTER',
       'CPAP/PSV+Apn TCPL', 'SIMV/VOL', 'NIV-ST', 'PRVC/SIMV', 'P-SIMV',
       'DuoPaP', 'APRV/Biphasic+ApnPress', 'SIMV/AutoFlow', 'nCPAP-PS',
       'APV (cmv)', 'ASV', 'CMV/ASSIST', 'SYNCHRON SLAVE',
       'CMV/ASSIST/AutoFlow', 'CPAP/PSV', 'MMV/PSV/AutoFlow', 'PSV/SBT',
       'SPONT', 'Standby'], dtype=object)

In [307]:
# Categories to replace with broader terms
values_to_replace_simv = [
    'SIMV/PSV', 'SIMV/AutoFlow', 'SIMV/PSV/AutoFlow', 'SIMV', 'SIMV/VOL',
    'P-SIMV', 'PRVC/SIMV', 'SIMV/PRES', 'APV (simv)'
]
values_to_replace_cmv = [
    'APV (cmv)', 'CMV/ASSIST', 'CMV/ASSIST/AutoFlow', 'CMV/AutoFlow',
    'P-CMV', 'CMV', '(S) CMV'
]
values_to_replace_psv = [
    'CPAP/PSV', 'PSV/SBT', 'MMV/PSV', 'MMV/PSV/AutoFlow', 'CPAP/PSV+ApnVol',
    'CPAP/PSV+ApnPres', 'PCV+/PSV', 'nCPAP-PS', 'CPAP/PSV+Apn TCPL'
]
values_to_replace_cpap = [
    'CPAP', 'CPAP/PPS', 'BiPAP/CPAP'
]
values_to_replace_aprv = [
    'APRV', 'APRV/Biphasic+ApnVol', 'APRV/Biphasic+ApnPress', 'DuoPaP'
]

# Creating replacement dictionaries
replacement_dict_simv = {value: 'SIMV' for value in values_to_replace_simv}
replacement_dict_cmv = {value: 'CMV' for value in values_to_replace_cmv}
replacement_dict_psv = {value: 'PSV' for value in values_to_replace_psv}
replacement_dict_cpap = {value: 'CPAP' for value in values_to_replace_cpap}
replacement_dict_aprv = {value: 'APRV' for value in values_to_replace_aprv}

# Combining all replacement dictionaries into one
replacement_dict = {**replacement_dict_simv, **replacement_dict_cmv, **replacement_dict_psv,
                    **replacement_dict_cpap, **replacement_dict_aprv}

# Replace the values in the 'value' column of your DataFrame
df_Ventilator['value'] = df_Ventilator['value'].replace(replacement_dict)

df_Ventilator['value'].unique()

/tmp/ipython-input-3606894189.py:33: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_Ventilator['value'] = df_Ventilator['value'].replace(replacement_dict)


array(['CMV', 'APRV', 'CPAP', 'VOL/AC', 'PRVC/AC', 'PSV', 'PCV+Assist',
       'SIMV', 'VS', 'NIV', 'Ambient', 'PCV+', 'MMV', 'PRES/AC',
       'MMV/AutoFlow', 'Apnea Ventilation', 'SYNCHRON MASTER', 'NIV-ST',
       'ASV', 'SYNCHRON SLAVE', 'SPONT', 'Standby'], dtype=object)

In [308]:
venitilator_sbt_list=['PSV','SPONT','CPAP','SIMV','APRV']
df_Ventilated_patients=df_Ventilator[df_Ventilator['value'].isin(venitilator_sbt_list)]
# df_Ventilator=df_Ventilator[df_Ventilator['itemid'].isin(if itemid in [223849, 229314] and value == 'Yes':)]

In [309]:
df_Ventilated_patients['value'].unique()

array(['APRV', 'CPAP', 'PSV', 'SIMV', 'SPONT'], dtype=object)

In [310]:
df_Ventilated_patients.head()

,subject_id,hadm_id,stay_id,caregiver_id,charttime,storetime,itemid,value,valuenum,valueuom,warning
13,19277757,28945718,32218208,267,2131-08-13 12:00:00,2131-08-13 12:35:00,223849,APRV,26.0,None,1
14,13993961,23570832,32890024,90569,2156-09-25 03:00:00,2156-09-25 03:32:00,223849,CPAP,10.0,None,0
15,13140343,27308490,39571846,23663,2183-01-02 05:00:00,2183-01-02 04:46:00,223849,CPAP,10.0,None,0
19,19787831,29586004,34188353,95176,2176-07-24 04:00:00,2176-07-24 04:55:00,223849,PSV,45.0,None,0
23,10017531,22580355,35526828,40326,2159-10-20 08:00:00,2159-10-20 08:08:00,223849,PSV,NaN,None,0


In [311]:
sbt_related_itemids = [
    224715,  # SBT Started
    224716,  # SBT Stopped
    224717,  # SBT Successfully Completed
    224833,  # SBT Deferred
]

In [312]:
query = f"""
SELECT *
FROM `{dataset_id}.mimiciv_3_1_icu.chartevents`
WHERE itemid IN ({', '.join(map(str, sbt_related_itemids))})
"""
df_sbt = client.query(query).to_dataframe()
df_sbt.head()

,subject_id,hadm_id,stay_id,caregiver_id,charttime,storetime,itemid,value,valuenum,valueuom,warning
0,16099332,29254134,36014754,72896,2168-11-24 04:02:00,2168-11-24 04:02:00,224833,Pending procedure,NaN,None,0
1,13135946,22133572,34063284,37909,2143-08-18 03:00:00,2143-08-18 03:12:00,224833,Pending procedure,NaN,None,0
2,16283147,21027423,38001523,52016,2134-03-10 04:43:00,2134-03-10 04:43:00,224833,Unable to perform RSBI,NaN,None,0
3,17733544,21643035,34345232,8083,2173-09-28 03:00:00,2173-09-28 03:38:00,224833,Unable to perform RSBI,NaN,None,0
4,15258544,24353503,30426514,6322,2158-03-02 04:00:00,2158-03-02 04:17:00,224833,Unable to perform RSBI,NaN,None,0


In [313]:
sbt_related_itemids_with_descriptions = {
    224715: "SBT Started",                                    # SBT Started
    224716: "SBT Stopped",                                    # SBT Stopped
    224717: "SBT Successfully Completed",                     # SBT Completed
    224833: "SBT Deferred",                                    # SBT Deferred
}

sbt_dict = {}

# Iterate over each itemid in the prioritized list with descriptions
for itemid, description in sbt_related_itemids_with_descriptions.items():
    # Filter the DataFrame for rows with the current itemid
    filtered_df = df_sbt[df_sbt['itemid'] == itemid]

    # Get the unique values for these rows and their value counts
    unique_values = filtered_df['value'].value_counts().to_dict()

    # Assign a dictionary with description, unique values, and their counts to the itemid in the final dictionary
    sbt_dict[itemid] = {"data description": description, "data unique values": unique_values}

sbt_dict

{224715: {'data description': 'SBT Started',
  'data unique values': {'Yes': 39729, 'No': 19145}},
 224716: {'data description': 'SBT Stopped',
  'data unique values': {'RR > 35 for > 5 min': 2872,
   'Accessory muscle use': 1660,
   'SpO2 < 90% for > 2 min': 966,
   'BP > 180 or < 90': 396,
   'HR > 140': 218,
   'Arrythmia': 138}},
 224717: {'data description': 'SBT Successfully Completed',
  'data unique values': {'Yes': 29610, 'No': 6085}},
 224833: {'data description': 'SBT Deferred',
  'data unique values': {'Unable to perform RSBI': 16936,
   'Chronic vent dependent patient': 3540,
   'Pending procedure ': 2460,
   'RSBI > 105': 349}}}

In [314]:
df_sbt_started=df_sbt[df_sbt['itemid']==224715]
df_sbt_started['value'].unique()

array(['No', 'Yes'], dtype=object)

In [315]:
df_sbt_started[df_sbt_started['value']=='Yes'].head()

,subject_id,hadm_id,stay_id,caregiver_id,charttime,storetime,itemid,value,valuenum,valueuom,warning
50,15796335,28186962,33835780,5663,2130-02-15 04:36:00,2130-02-15 05:01:00,224715,Yes,NaN,None,0
51,19216175,23788838,32119996,25780,2134-03-17 11:00:00,2134-03-17 14:13:00,224715,Yes,NaN,None,1
52,16026359,23809913,36714535,45039,2136-11-26 03:45:00,2136-11-26 03:51:00,224715,Yes,NaN,None,0
53,15535502,29603866,39075951,37909,2127-11-25 05:00:00,2127-11-25 05:09:00,224715,Yes,NaN,None,0
54,14902020,26535506,34234630,68397,2114-02-10 04:53:00,2114-02-10 04:54:00,224715,Yes,NaN,None,0


In [316]:
len(df_sbt_started[df_sbt_started['value']=='Yes']['subject_id'].unique())

13961

In [317]:
df_sbt_stopped=df_sbt[df_sbt['itemid']==224716]
len(df_sbt_stopped['subject_id'].unique())

2887

In [318]:
df_sbt_stopped['value'].unique()

array(['Arrythmia', 'RR > 35 for > 5 min', 'Accessory muscle use',
       'SpO2 < 90% for > 2 min', 'BP > 180 or < 90', 'HR > 140'],
      dtype=object)

In [319]:
df_sbt_deferred=df_sbt[df_sbt['itemid']==224833]
len(df_sbt_deferred['subject_id'].unique())

7941

In [320]:
df_sbt_success=df_sbt[df_sbt['itemid']==224717]
df_sbt_success.head()

,subject_id,hadm_id,stay_id,caregiver_id,charttime,storetime,itemid,value,valuenum,valueuom,warning
89,12832775,24214772,34978160,31733,2114-12-07 05:00:00,2114-12-07 04:43:00,224717,No,NaN,None,0
90,19404265,26466590,36121575,9698,2175-07-24 03:00:00,2175-07-24 03:10:00,224717,No,NaN,None,1
91,13509204,29281474,34588667,80858,2134-12-05 05:00:00,2134-12-05 05:28:00,224717,No,NaN,None,0
92,13678189,20731516,33368602,94373,2142-10-21 03:50:00,2142-10-21 03:54:00,224717,No,NaN,None,0
93,10729759,21158396,33398332,7630,2134-08-23 05:11:00,2134-08-23 05:12:00,224717,Yes,NaN,None,1


In [321]:
df_sbt_success['value'].unique()
len(df_sbt_success['subject_id'].unique())

13137

In [322]:
len(df_sbt_success[df_sbt_success['value']=='Yes']['subject_id'].unique())

12293

# Component C

In [323]:
regex= r"(?i)\bRichmond[-\s]*RAS{1,2}[-\s]*Scale\b"
regex_output= client.query(f"""
SELECT *
FROM `{dataset_id}.mimiciv_3_1_icu.d_items`
WHERE REGEXP_CONTAINS(
  LOWER(CONCAT(' ',
    IFNULL(label,''),' ',
    IFNULL(abbreviation,''),' '
  )),
  r\"\"\"{regex}\"\"\"
)
""").to_dataframe()
regex_output

,itemid,label,abbreviation,linksto,category,unitname,param_type,lownormalvalue,highnormalvalue
0,228096,Richmond-RAS Scale,Richmond-RAS Scale,chartevents,Pain/Sedation,None,Text,NaN,NaN
1,228299,Goal Richmond-RAS Scale,Goal Richmond-RAS Scale,chartevents,Pain/Sedation,None,Text,NaN,NaN


In [324]:
regex= r"\b(?:analgesic(?:s)?|analgesia)\b(?:\s*[:/-])?"
# regex= r"\b(?:sedative(?:s)?|sedation)\b(?:\s*[:/-])?"
regex_output= client.query(f"""
SELECT *
FROM `{dataset_id}.mimiciv_3_1_icu.d_items`
WHERE REGEXP_CONTAINS(
  LOWER(CONCAT(' ',
    IFNULL(label,''),' ',
    IFNULL(abbreviation,''),' '
  )),
  r\"\"\"{regex}\"\"\"
)
""").to_dataframe()
regex_output

,itemid,label,abbreviation,linksto,category,unitname,param_type,lownormalvalue,highnormalvalue
0,230117,Analgesics,Analgesics,chartevents,Block Charting Note,None,Text,NaN,NaN


In [325]:
regex= r"\b(?:sedative(?:s)?|sedation)\b(?:\s*[:/-])?"
regex_output= client.query(f"""
SELECT *
FROM `{dataset_id}.mimiciv_3_1_icu.d_items`
WHERE REGEXP_CONTAINS(
  LOWER(CONCAT(' ',
    IFNULL(label,''),' ',
    IFNULL(abbreviation,''),' '
  )),
  r\"\"\"{regex}\"\"\"
)
""").to_dataframe()
regex_output

,itemid,label,abbreviation,linksto,category,unitname,param_type,lownormalvalue,highnormalvalue
0,230116,Sedatives,Sedatives,chartevents,Block Charting Note,None,Text,NaN,NaN
1,228060,Conscious sedation used (Bronch),Conscious sedation used (Bronch),chartevents,Bronchoscopy,None,Text,NaN,NaN
2,225531,Conscious sedation used (Bronch),Conscious sedation used (Bronch),chartevents,Bronchoscopy,None,Checkbox,NaN,NaN
3,224506,Conscious sedation (THCEN),Conscious sedation (THCEN),chartevents,Thoracentesis,None,Checkbox,NaN,NaN
4,228710,Sedation goal,Sedation goal,chartevents,MD Progress Note,None,Text,NaN,NaN
5,228473,Conscious Sedation (Gen Proc),Conscious Sedation (Gen Proc),chartevents,Generic Proc Note,None,Text,NaN,NaN


**Alternative Way to Extract Concepts**

In [326]:
ras_d_items_filtered = client.query(f'''
SELECT * FROM `{dataset_id}.mimiciv_3_1_icu.d_items`
WHERE label LIKE '%%Richmond%%'
OR abbreviation LIKE '%%RAS%%'
OR abbreviation LIKE '%%Sedative%%'
OR abbreviation LIKE '%%Analgesic%%'
''').to_dataframe()
ras_d_items_filtered

,itemid,label,abbreviation,linksto,category,unitname,param_type,lownormalvalue,highnormalvalue
0,230116,Sedatives,Sedatives,chartevents,Block Charting Note,None,Text,NaN,NaN
1,230117,Analgesics,Analgesics,chartevents,Block Charting Note,None,Text,NaN,NaN
2,228096,Richmond-RAS Scale,Richmond-RAS Scale,chartevents,Pain/Sedation,None,Text,NaN,NaN
3,228299,Goal Richmond-RAS Scale,Goal Richmond-RAS Scale,chartevents,Pain/Sedation,None,Text,NaN,NaN
4,228302,CAM-ICU RASS LOC,CAM-ICU RASS LOC,chartevents,Pain/Sedation,None,Text,NaN,NaN


In [327]:
ras_relevant_itemids = [
    228096,  # Richmond-RAS Scale
    228299,  # Goal Richmond-RAS Scale
    230116,  # Sedatives
    230117,  # Analgesics
         ]
query = f"""
SELECT *
FROM `{dataset_id}.mimiciv_3_1_icu.chartevents`
WHERE itemid IN ({', '.join(map(str, ras_relevant_itemids))})
"""
df_RASS = client.query(query).to_dataframe()
df_RASS.head(2)

,subject_id,hadm_id,stay_id,caregiver_id,charttime,storetime,itemid,value,valuenum,valueuom,warning
0,15556698,26628429,34394728,39083,2169-02-25 00:00:00,2169-02-24 23:08:00,228096,0 Alert and calm,0.0,None,0
1,17869052,20177578,37663803,93093,2117-07-13 12:00:00,2117-07-13 11:46:00,228096,0 Alert and calm,0.0,None,0


In [328]:
ras_relevant_itemids = {
    228096:  "Richmond-RAS Scale",
    228299:  "Goal Richmond-RAS Scale",
    230116:  "Sedatives",
    230117:  "Analgesics"
         }

c_dict = {}

# Iterate over each itemid in the prioritized list with descriptions
for itemid, description in ras_relevant_itemids.items():
    # Filter the DataFrame for rows with the current itemid
    filtered_df = df_RASS[df_RASS['itemid'] == itemid]

    # Get the unique values for these rows and their value counts
    unique_values = filtered_df['value'].value_counts().to_dict()

    # Assign a dictionary with description, unique values, and their counts to the itemid in the final dictionary
    c_dict[itemid] = {"data description": description, "data unique values": unique_values}

c_dict

{228096: {'data description': 'Richmond-RAS Scale',
  'data unique values': {' 0  Alert and calm': 787684,
   '-1 Awakens to voice (eye opening/contact) > 10 sec': 279320,
   '-2 Light sedation, briefly awakens to voice (eye opening/contact) < 10 sec': 124113,
   '+1 Anxious, apprehensive, but not aggressive': 121062,
   '-3 Moderate sedation, movement or eye opening; No eye contact': 108251,
   '-4 Deep sedation, no response to voice, but movement or eye opening to physical stimulation': 95658,
   '-5 Unarousable, no response to voice or physical stimulation': 90411,
   '+2 Frequent nonpurposeful movement, fights ventilator': 33011,
   '+3 Pulls or removes tube(s) or catheter(s); aggressive': 14043,
   '+4 Combative, violent, danger to staff': 2448}},
 228299: {'data description': 'Goal Richmond-RAS Scale',
  'data unique values': {' 0  Alert and calm': 993599,
   '-1 Awakens to voice (eye opening/contact) > 10 sec': 252047,
   '-5 Unarousable, no response to voice or physical stimula

In [329]:
df_RASS[df_RASS['itemid']==228096]['value'].unique()

array([' 0  Alert and calm', '+4 Combative, violent, danger to staff',
       '+1 Anxious, apprehensive, but not aggressive',
       '-1 Awakens to voice (eye opening/contact) > 10 sec',
       '+2 Frequent nonpurposeful movement, fights ventilator',
       '-5 Unarousable, no response to voice or physical stimulation',
       '-3 Moderate sedation, movement or eye opening; No eye contact',
       '-2 Light sedation, briefly awakens to voice (eye opening/contact) < 10 sec',
       '-4 Deep sedation, no response to voice, but movement or eye opening to physical stimulation',
       '+3 Pulls or removes tube(s) or catheter(s); aggressive'],
      dtype=object)

In [330]:
df_RASS[df_RASS['itemid']==228299]['value'].unique()

array([' 0  Alert and calm',
       '-1 Awakens to voice (eye opening/contact) > 10 sec',
       '-5 Unarousable, no response to voice or physical stimulation',
       '-3 Moderate sedation, movement or eye opening; No eye contact',
       '-2 Light sedation, briefly awakens to voice (eye opening/contact) < 10 sec',
       '-4 Deep sedation, no response to voice, but movement or eye opening to physical stimulation'],
      dtype=object)

In [331]:
df_RASS['date'] = pd.to_datetime(df_RASS['charttime']).dt.date

In [332]:
len(df_RASS['date'].unique())

35505

In [333]:
df_RASS_goal_set = df_RASS[df_RASS['itemid'] == 228299]
df_RASS_goal_set['subject_id'].nunique()

54087

In [334]:
assessment_itemid = 228096  # Richmond-RAS Scale
df_RASS_assessments = df_RASS[df_RASS['itemid'] == assessment_itemid]
df_RASS_assessments.head(3)

,subject_id,hadm_id,stay_id,caregiver_id,charttime,storetime,itemid,value,valuenum,valueuom,warning,date
0,15556698,26628429,34394728,39083,2169-02-25 00:00:00,2169-02-24 23:08:00,228096,0 Alert and calm,0.0,None,0,2169-02-25
1,17869052,20177578,37663803,93093,2117-07-13 12:00:00,2117-07-13 11:46:00,228096,0 Alert and calm,0.0,None,0,2117-07-13
2,14344243,29899161,33302469,38240,2110-05-29 16:00:00,2110-05-29 15:58:00,228096,0 Alert and calm,0.0,None,0,2110-05-29


In [335]:
df_RASS_assessments['subject_id'].nunique()

58770

In [336]:
assessments_per_day = df_RASS_assessments.groupby(['subject_id', 'date']).size().reset_index(name='counts')
assessments_per_day['adherent'] = assessments_per_day['counts'] >= 6
assessments_per_day.head()

,subject_id,date,counts,adherent
0,10000032,2180-07-23,2,False
1,10000980,2189-06-27,3,False
2,10001217,2157-11-21,6,True
3,10001217,2157-12-19,2,False
4,10001217,2157-12-20,4,False


In [337]:
adherent_df = assessments_per_day[assessments_per_day['adherent'] == True]
unique_subject_ids_true_adherent = adherent_df['subject_id'].unique()
print("Unique subject IDs with true adherence:", len(unique_subject_ids_true_adherent))

Unique subject IDs with true adherence: 33448


In [338]:
adherence_percentage = assessments_per_day.groupby('subject_id')['adherent'].mean() * 100
average_adherence = adherence_percentage.mean()
print(f"Average Adherence Percentage (days with 6 or more assessments): {average_adherence:.2f}%")


Average Adherence Percentage (days with 6 or more assessments): 25.63%


In [339]:
assessments_per_day = assessments_per_day.reset_index()
aggregated_data = assessments_per_day.groupby('date').sum()

subjects_with_228096 = df_RASS[df_RASS['itemid'] == 228096]['subject_id'].unique()
subjects_with_228299 = df_RASS[df_RASS['itemid'] == 228299]['subject_id'].unique()
intersection_subjects = np.setdiff1d(subjects_with_228096, subjects_with_228299)

print("Unique subjects with itemid 228096 (Richmond-RAS Scale):", len(subjects_with_228096))
print("Unique subjects with itemid 228299 (Goal Richmond-RAS Scale):", len(subjects_with_228299))
print("Subjects with not in both assessments:", len(intersection_subjects))

Unique subjects with itemid 228096 (Richmond-RAS Scale): 58770
Unique subjects with itemid 228299 (Goal Richmond-RAS Scale): 54087
Subjects with not in both assessments: 4685


In [340]:
# Assessments per patient per day
assessments_per_day = df_RASS_assessments.groupby(['subject_id', 'date']).size().reset_index(name='counts')
compliant_days = assessments_per_day[assessments_per_day['counts'] >= 6]
compliance_rate = (compliant_days.shape[0] / assessments_per_day.shape[0]) * 100

# Median and IQR for assessments per day
median_assessments = assessments_per_day['counts'].median()
iqr_assessments = assessments_per_day['counts'].quantile(0.75) - assessments_per_day['counts'].quantile(0.25)

# Mean and standard deviation for assessments per day
mean_assessments = assessments_per_day['counts'].mean()
std_assessments = assessments_per_day['counts'].std()

# Minimum and maximum assessments per day
min_assessments = assessments_per_day['counts'].min()
max_assessments = assessments_per_day['counts'].max()

print("Compliance Rate (%):", compliance_rate)
print("Median assessments per day:", median_assessments)
print("Interquartile range (IQR):", iqr_assessments)
print("Mean assessments per day:", mean_assessments)
print("Standard deviation (SD):", std_assessments)
print("Minimum assessments per day:", min_assessments)
print("Maximum assessments per day:", max_assessments)

Compliance Rate (%): 37.72224455297803
Median assessments per day: 5.0
Interquartile range (IQR): 3.0
Mean assessments per day: 4.559147748233882
Standard deviation (SD): 2.1822728370752844
Minimum assessments per day: 1
Maximum assessments per day: 25


In [341]:
assessments_per_day = df_RASS_assessments.groupby(['subject_id', 'date']).size().reset_index(name='counts')
compliant_days = assessments_per_day[assessments_per_day['counts'] >= 6]
compliant_days.head(3)


,subject_id,date,counts
2,10001217,2157-11-21,6
10,10001884,2131-01-12,6
12,10001884,2131-01-14,6


In [342]:
compliant_days.shape[0]

137017

In [343]:
compliant_days['subject_id'].nunique()

33448

In [344]:
len(assessments_per_day['subject_id'].unique())

58770

# Component D

In [345]:
regex= r"(?i)\bCAM[-\s]*ICU[-\s]+RASS[-\s]+LOC\b"
regex_output= client.query(f"""
SELECT *
FROM `{dataset_id}.mimiciv_3_1_icu.d_items`
WHERE REGEXP_CONTAINS(
  LOWER(CONCAT(' ',
    IFNULL(label,''),' ',
    IFNULL(abbreviation,''),' '
  )),
  r\"\"\"{regex}\"\"\"
)
""").to_dataframe()
regex_output

,itemid,label,abbreviation,linksto,category,unitname,param_type,lownormalvalue,highnormalvalue
0,228302,CAM-ICU RASS LOC,CAM-ICU RASS LOC,chartevents,Pain/Sedation,None,Text,NaN,NaN


In [346]:
regex= r"(?i)\bCAM[-\s]*ICU[-\s]+ms[-\s]+change\b"
regex_output= client.query(f"""
SELECT *
FROM `{dataset_id}.mimiciv_3_1_icu.d_items`
WHERE REGEXP_CONTAINS(
  LOWER(CONCAT(' ',
    IFNULL(label,''),' ',
    IFNULL(abbreviation,''),' '
  )),
  r\"\"\"{regex}\"\"\"
)
""").to_dataframe()
regex_output

,itemid,label,abbreviation,linksto,category,unitname,param_type,lownormalvalue,highnormalvalue
0,228300,CAM-ICU MS change,CAM-ICU MS change,chartevents,Pain/Sedation,None,Text,NaN,NaN
1,228337,CAM-ICU MS Change,CAM-ICU MS Change,chartevents,Pain/Sedation,None,Text,NaN,NaN
2,229326,CAM-ICU MS Change,CAM-ICU MS Change,chartevents,Pain/Sedation,None,Text,NaN,NaN


In [347]:
regex= r"(?i)\bCAM[-\s]*ICU[-\s]+inattention\b"
regex_output= client.query(f"""
SELECT *
FROM `{dataset_id}.mimiciv_3_1_icu.d_items`
WHERE REGEXP_CONTAINS(
  LOWER(CONCAT(' ',
    IFNULL(label,''),' ',
    IFNULL(abbreviation,''),' '
  )),
  r\"\"\"{regex}\"\"\"
)
""").to_dataframe()
regex_output

,itemid,label,abbreviation,linksto,category,unitname,param_type,lownormalvalue,highnormalvalue
0,228301,CAM-ICU Inattention,CAM-ICU Inattention,chartevents,Pain/Sedation,None,Text,NaN,NaN
1,228336,CAM-ICU Inattention,CAM-ICU Inattention,chartevents,Pain/Sedation,None,Text,NaN,NaN
2,229325,CAM-ICU Inattention,CAM-ICU Inattention,chartevents,Pain/Sedation,None,Text,NaN,NaN


In [348]:
regex= r"(?i)\bCAM[-\s]*ICU[-\s]+altered[-\s]+LOC\b"
regex_output= client.query(f"""
SELECT *
FROM `{dataset_id}.mimiciv_3_1_icu.d_items`
WHERE REGEXP_CONTAINS(
  LOWER(CONCAT(' ',
    IFNULL(label,''),' ',
    IFNULL(abbreviation,''),' '
  )),
  r\"\"\"{regex}\"\"\"
)
""").to_dataframe()
regex_output

,itemid,label,abbreviation,linksto,category,unitname,param_type,lownormalvalue,highnormalvalue
0,228334,CAM-ICU Altered LOC,CAM-ICU Altered LOC,chartevents,Pain/Sedation,None,Text,NaN,NaN


In [349]:
regex= r"(?i)\bCAM[-\s]*ICU[-\s]+disorgani[sz]ed[-\s]+thinking\b"
regex_output= client.query(f"""
SELECT *
FROM `{dataset_id}.mimiciv_3_1_icu.d_items`
WHERE REGEXP_CONTAINS(
  LOWER(CONCAT(' ',
    IFNULL(label,''),' ',
    IFNULL(abbreviation,''),' '
  )),
  r\"\"\"{regex}\"\"\"
)
""").to_dataframe()
regex_output

,itemid,label,abbreviation,linksto,category,unitname,param_type,lownormalvalue,highnormalvalue
0,228303,CAM-ICU Disorganized thinking,CAM-ICU Disorganized thinking,chartevents,Pain/Sedation,None,Text,NaN,NaN
1,228335,CAM-ICU Disorganized thinking,CAM-ICU Disorganized thinking,chartevents,Pain/Sedation,None,Text,NaN,NaN
2,229324,CAM-ICU Disorganized thinking,CAM-ICU Disorganized thinking,chartevents,Pain/Sedation,None,Text,NaN,NaN


In [350]:
regex= r"(?i)\b(?:delirium(?:s)?|delirium[-\s]+assessment[-\s]+status)\b"
regex_output= client.query(f"""
SELECT *
FROM `{dataset_id}.mimiciv_3_1_icu.d_items`
WHERE REGEXP_CONTAINS(
  LOWER(CONCAT(' ',
    IFNULL(label,''),' ',
    IFNULL(abbreviation,''),' '
  )),
  r\"\"\"{regex}\"\"\"
)
""").to_dataframe()
regex_output

,itemid,label,abbreviation,linksto,category,unitname,param_type,lownormalvalue,highnormalvalue
0,228332,Delirium assessment,Delirium assessment,chartevents,Pain/Sedation,None,Text,NaN,NaN
1,228688,Delirium,Delirium,chartevents,MD Progress Note,None,Text,NaN,NaN


In [351]:
regex=r"(?i)\b(?:CAM[-\s]*ICU(?:[-\s]+(?:RASS[-\s]+LOC|ms[-\s]+change|inattention|altered[-\s]+LOC|disorgani[sz]ed[-\s]+thinking))|deliriums?|delirium[-\s]+assessment[-\s]+status)\b"
regex_output= client.query(f"""
SELECT *
FROM `{dataset_id}.mimiciv_3_1_icu.d_items`
WHERE REGEXP_CONTAINS(
  LOWER(CONCAT(' ',
    IFNULL(label,''),' ',
    IFNULL(abbreviation,''),' '
  )),
  r\"\"\"{regex}\"\"\"
)
""").to_dataframe()
regex_output


,itemid,label,abbreviation,linksto,category,unitname,param_type,lownormalvalue,highnormalvalue
0,228300,CAM-ICU MS change,CAM-ICU MS change,chartevents,Pain/Sedation,None,Text,NaN,NaN
1,228301,CAM-ICU Inattention,CAM-ICU Inattention,chartevents,Pain/Sedation,None,Text,NaN,NaN
2,228302,CAM-ICU RASS LOC,CAM-ICU RASS LOC,chartevents,Pain/Sedation,None,Text,NaN,NaN
3,228303,CAM-ICU Disorganized thinking,CAM-ICU Disorganized thinking,chartevents,Pain/Sedation,None,Text,NaN,NaN
4,228332,Delirium assessment,Delirium assessment,chartevents,Pain/Sedation,None,Text,NaN,NaN
5,228334,CAM-ICU Altered LOC,CAM-ICU Altered LOC,chartevents,Pain/Sedation,None,Text,NaN,NaN
6,228335,CAM-ICU Disorganized thinking,CAM-ICU Disorganized thinking,chartevents,Pain/Sedation,None,Text,NaN,NaN
7,228336,CAM-ICU Inattention,CAM-ICU Inattention,chartevents,Pain/Sedation,None,Text,NaN,NaN
8,228337,CAM-ICU MS Change,CAM-ICU MS Change,chartevents,Pain/Sedation,None,Text,NaN,NaN
9,229324,CAM-ICU Disorganized thinking,CAM-ICU Disorganized thinking,chartevents,Pain/Sedation,None,Text,NaN,NaN


**Alternative Way to Extract Concepts**

In [352]:
delirium_d_items_filtered = client.query(f'''
SELECT * FROM `{dataset_id}.mimiciv_3_1_icu.d_items`
WHERE label LIKE '%%CAM-ICU%%'
OR abbreviation LIKE '%%Delirium%%'
''').to_dataframe()
delirium_d_items_filtered

,itemid,label,abbreviation,linksto,category,unitname,param_type,lownormalvalue,highnormalvalue
0,228300,CAM-ICU MS change,CAM-ICU MS change,chartevents,Pain/Sedation,None,Text,NaN,NaN
1,228301,CAM-ICU Inattention,CAM-ICU Inattention,chartevents,Pain/Sedation,None,Text,NaN,NaN
2,228302,CAM-ICU RASS LOC,CAM-ICU RASS LOC,chartevents,Pain/Sedation,None,Text,NaN,NaN
3,228303,CAM-ICU Disorganized thinking,CAM-ICU Disorganized thinking,chartevents,Pain/Sedation,None,Text,NaN,NaN
4,228332,Delirium assessment,Delirium assessment,chartevents,Pain/Sedation,None,Text,NaN,NaN
5,228334,CAM-ICU Altered LOC,CAM-ICU Altered LOC,chartevents,Pain/Sedation,None,Text,NaN,NaN
6,228335,CAM-ICU Disorganized thinking,CAM-ICU Disorganized thinking,chartevents,Pain/Sedation,None,Text,NaN,NaN
7,228336,CAM-ICU Inattention,CAM-ICU Inattention,chartevents,Pain/Sedation,None,Text,NaN,NaN
8,228337,CAM-ICU MS Change,CAM-ICU MS Change,chartevents,Pain/Sedation,None,Text,NaN,NaN
9,229324,CAM-ICU Disorganized thinking,CAM-ICU Disorganized thinking,chartevents,Pain/Sedation,None,Text,NaN,NaN


In [353]:
delirium_assessment_itemids = [
    228301, # CAM-ICU MS change
    228302, # CAM-ICU Inattention
    228303, # CAM-ICU Disorganized thinking
    228332, # Delirium assessment
    228334, # CAM-ICU Altered LOC
    228335, # CAM-ICU Disorganized thinking
    228336, # CAM-ICU Inattention
    228337, # CAM-ICU MS Changef
    228688, # Delirium
    229324, # CAM-ICU Disorganized thinking (Hamilton)
    229325, # CAM-ICU Inattention (Hamilton)
    229326  # CAM-ICU MS Change (Hamilton)
]
query = f"""
SELECT *
FROM `{dataset_id}.mimiciv_3_1_icu.chartevents`
WHERE itemid IN ({', '.join(map(str, delirium_assessment_itemids))})
"""
df_delirium = client.query(query).to_dataframe()
df_delirium.head()

,subject_id,hadm_id,stay_id,caregiver_id,charttime,storetime,itemid,value,valuenum,valueuom,warning
0,11871434,22356021,38573166,60065,2144-10-09 20:38:00,2144-10-09 20:38:00,229325,Language Barrier,0.0,None,0
1,19076225,21136805,39865389,51807,2187-08-05 20:00:00,2187-08-06 00:25:00,229325,Language Barrier,0.0,None,0
2,12878814,28573817,36320609,91925,2116-10-17 08:00:00,2116-10-17 10:30:00,229325,Language Barrier,0.0,None,0
3,12384505,23489615,38928375,25139,2147-11-07 08:00:00,2147-11-07 08:30:00,229325,Language Barrier,0.0,None,0
4,17436451,23236799,39494479,77753,2147-08-07 08:00:00,2147-08-07 09:11:00,229325,Preexisting Advanced Dementia,0.0,None,0


In [354]:
df_delirium['date'] = pd.to_datetime(df_delirium['charttime']).dt.date
df_delirium_assesments=df_delirium[df_delirium['itemid']!=228332]
df_delirium_satatus=df_delirium[df_delirium['itemid']==228332]
df_delirium_assesments['itemid'].unique()

<IntegerArray>
[229325, 228301, 229326, 228303, 228334, 228335, 228336, 228337, 229324,
 228302]
Length: 10, dtype: Int64

In [355]:
df_delirium_assesments['subject_id'].nunique()

51579

In [356]:
daily_counts = df_delirium_assesments.groupby(['subject_id', 'date']).size().reset_index(name='counts')
daily_counts['date'].nunique()

35015

In [357]:
daily_counts['adherent'] = daily_counts['counts'] >= 2
daily_compliance = daily_counts.groupby('date')['adherent'].mean() * 100
average_daily_compliance = daily_compliance.mean()
print(f"Average Daily Compliance Percentage: {average_daily_compliance:.2f}%")

Average Daily Compliance Percentage: 71.61%


In [358]:
adherent_df = daily_counts[daily_counts['adherent'] == True]
unique_subject_ids_true_adherent = adherent_df['subject_id'].unique()
print("Unique subject IDs with true adherence:", len(unique_subject_ids_true_adherent))

Unique subject IDs with true adherence: 44142


In [359]:
daily_status_counts = df_delirium_satatus.groupby(['subject_id', 'date','value']).size().reset_index(name='counts')
delirium_present = daily_status_counts[daily_status_counts['value'] == 'Positive']
delirium_present= delirium_present['subject_id'].unique()
print("Unique subject IDs with true adherence:", len(delirium_present))

Unique subject IDs with true adherence: 18180


In [360]:
# Delirium assessments
daily_counts = df_delirium_assesments.groupby(['subject_id', 'date']).size().reset_index(name='counts')

# Compliant days (where assessments >= 2)
daily_counts['adherent'] = daily_counts['counts'] >= 2

# Compliance percentage (average daily compliance)
compliance_rate = (daily_counts['adherent'].mean()) * 100

# Median and IQR for delirium assessments per day
median_assessments = daily_counts['counts'].median()
iqr_assessments = daily_counts['counts'].quantile(0.75) - daily_counts['counts'].quantile(0.25)

# Mean and standard deviation for delirium assessments per day
mean_assessments = daily_counts['counts'].mean()
std_assessments = daily_counts['counts'].std()

# Minimum and maximum assessments per day
min_assessments = daily_counts['counts'].min()
max_assessments = daily_counts['counts'].max()

# Number of unique patients with true adherence
adherent_df = daily_counts[daily_counts['adherent'] == True]
unique_subject_ids_true_adherent = adherent_df['subject_id'].unique()

print("Compliance Rate (%):", compliance_rate)
print("Median assessments per day:", median_assessments)
print("Interquartile range (IQR):", iqr_assessments)
print("Mean assessments per day:", mean_assessments)
print("Standard deviation (SD):", std_assessments)
print("Minimum assessments per day:", min_assessments)
print("Maximum assessments per day:", max_assessments)
print("Unique subject IDs with true adherence:", len(unique_subject_ids_true_adherent))


Compliance Rate (%): 71.50433383466542
Median assessments per day: 2.0
Interquartile range (IQR): 3.0
Mean assessments per day: 3.1833225190624965
Standard deviation (SD): 2.6123009774293386
Minimum assessments per day: 1
Maximum assessments per day: 34
Unique subject IDs with true adherence: 44142


# Component E

In [361]:
regex= r"(?i)\bRange[-\s]+of[-\s]+Motion\b"
regex_output= client.query(f"""
SELECT *
FROM `{dataset_id}.mimiciv_3_1_icu.d_items`
WHERE REGEXP_CONTAINS(
  LOWER(CONCAT(' ',
    IFNULL(label,''),' ',
    IFNULL(abbreviation,''),' '
  )),
  r\"\"\"{regex}\"\"\"
)
""").to_dataframe()
regex_output

,itemid,label,abbreviation,linksto,category,unitname,param_type,lownormalvalue,highnormalvalue
0,224092,Range of Motion Location,ROM Location,chartevents,Treatments,None,Text,NaN,NaN
1,225176,Range of Motion Status,ROM Status,chartevents,Treatments,None,Text,NaN,NaN
2,224067,Range of Motion,Range of Motion,chartevents,Restraint/Support Systems,None,Text,NaN,NaN
3,227953,Range of Motion,Range of Motion,chartevents,Restraint/Support Systems,None,Text,NaN,NaN


In [362]:
regex= r"(?i)\b(?:Gait[-/\s]+Transfer+ing|Transfers?)\b"
regex_output= client.query(f"""
SELECT *
FROM `{dataset_id}.mimiciv_3_1_icu.d_items`
WHERE REGEXP_CONTAINS(
  LOWER(CONCAT(' ',
    IFNULL(label,''),' ',
    IFNULL(abbreviation,''),' '
  )),
  r\"\"\"{regex}\"\"\"
)
""").to_dataframe()
regex_output

,itemid,label,abbreviation,linksto,category,unitname,param_type,lownormalvalue,highnormalvalue
0,228136,Family notified of transfer,Family notified of transfer,procedureevents,7-Communication,None,Processes,NaN,NaN
1,227848,Transfer,Transfer,chartevents,OT Notes,None,Text,NaN,NaN
2,229579,Transfer Intercampus by Ambulance,Transfer Intercampus by Ambulance,procedureevents,3-Significant Events,None,Processes,NaN,NaN
3,227345,Gait/Transferring,Gait/Transferring,chartevents,Restraint/Support Systems,None,Text,NaN,NaN


In [363]:
regex= r"(?i)\bSit[-\s]*to[-\s]*Stand|\b"
regex_output= client.query(f"""
SELECT *
FROM `{dataset_id}.mimiciv_3_1_icu.d_items`
WHERE REGEXP_CONTAINS(
  LOWER(CONCAT(' ',
    IFNULL(label,''),' ',
    IFNULL(abbreviation,''),' '
  )),
  r\"\"\"{regex}\"\"\"
)
""").to_dataframe()
regex_output

,itemid,label,abbreviation,linksto,category,unitname,param_type,lownormalvalue,highnormalvalue
0,220120,Intra Aortic Ballon Pump Setting,Intra Aortic Ballon Pump Setting,chartevents,IABP,None,Text,NaN,NaN
1,223933,Heat Sounds,Heat Sounds,chartevents,Cardiovascular (Pulses),None,Text,NaN,NaN
2,223934,Dorsal PedPulse R,Dorsal PedPulse R,chartevents,Cardiovascular (Pulses),None,Text,NaN,NaN
3,223935,PostTib. Pulses R,PostTib Pulses R,chartevents,Cardiovascular (Pulses),None,Text,NaN,NaN
4,223936,Radial Pulse R,Radial Pulse R,chartevents,Cardiovascular (Pulses),None,Text,NaN,NaN
...,...,...,...,...,...,...,...,...,...
4090,221032,Lauric acid,Lauric acid,ingredientevents,Ingredients - general (Not In Use),mg,Ingredient,NaN,NaN
4091,221033,Palmitic acid,Palmitic acid,ingredientevents,Ingredients - general (Not In Use),mg,Ingredient,NaN,NaN
4092,221035,Oleic acid,Oleic acid,ingredientevents,Ingredients - general (Not In Use),mg,Ingredient,NaN,NaN
4093,221194,Hexastarch,Hexastarch,ingredientevents,Ingredients - general (Not In Use),mg,Ingredient,NaN,NaN


In [364]:
regex= r"(?i)\bMobilization[-\s]+Plan\b"
regex_output= client.query(f"""
SELECT *
FROM `{dataset_id}.mimiciv_3_1_icu.d_items`
WHERE REGEXP_CONTAINS(
  LOWER(CONCAT(' ',
    IFNULL(label,''),' ',
    IFNULL(abbreviation,''),' '
  )),
  r\"\"\"{regex}\"\"\"
)
""").to_dataframe()
regex_output

,itemid,label,abbreviation,linksto,category,unitname,param_type,lownormalvalue,highnormalvalue
0,228697,Mobilization Plan,Mobilization Plan,chartevents,MD Progress Note,None,Text,NaN,NaN


In [365]:
terms  = [
    "ambulation",
    "mobilization",
    "therapy",  # often used in conjunction with physical or occupational
    "ROM",  # abbreviation for Range of Motion
    "transfer",  # as in transfer training or bed to chair transfer
    "gait",  # gait training
    "stand",  # sit to stand exercises
    "positioning",  # as in upright positioning or bed positioning
    "rehabilitation"  # often used to describe the overall process of mobilization
]

pattern = r"(?i)\b(" + "|".join(re.escape(t) for t in terms) + r")\b"

sql = f"""
SELECT itemid, label, abbreviation, linksto, category, unitname
FROM `{dataset_id}.mimiciv_3_1_icu.d_items`
WHERE REGEXP_CONTAINS(
  CONCAT(
    ' ', IFNULL(label,''), ' ',
    IFNULL(abbreviation,''), ' ',
    IFNULL(category,''), ' ',
    IFNULL(unitname,''), ' ',
    IFNULL(linksto,''), ' '
  ),
  r'{pattern}'
)
"""

mobility_d_items_filtered = client.query(sql).to_dataframe()
mobility_d_items_filtered

,itemid,label,abbreviation,linksto,category,unitname
0,228136,Family notified of transfer,Family notified of transfer,procedureevents,7-Communication,None
1,227847,Sit to Stand,Sit to Stand,chartevents,OT Notes,None
2,227848,Transfer,Transfer,chartevents,OT Notes,None
3,227905,Patient agrees with the above goals and is wil...,Patient agrees with the above goals and is wil...,chartevents,OT Notes,None
4,225298,Neck ROM (Intubation),Neck ROM (Intubation),chartevents,Intubation,None
5,224092,Range of Motion Location,ROM Location,chartevents,Treatments,None
6,225176,Range of Motion Status,ROM Status,chartevents,Treatments,None
7,225450,Radiation Therapy,Radiation Therapy,procedureevents,4-Procedures,None
8,228061,Photodynamic therapy PDT (Bronch),Photodynamic therapy PDT (Bronch),chartevents,Bronchoscopy,None
9,225532,Photodynamic therapy PDT (Bronch),Photodynamic therapy PDT (Bronch),chartevents,Bronchoscopy,None


In [366]:
mobilization_related_itemids = [
    224092, # Range of Motion Location
    225176, # Range of Motion Status
    227345, # Gait/Transferring
    227847, # Sit to Stand
    227848, # Transfer
    228697, # Mobilization Plan
]
query = f"""
SELECT *
FROM `{dataset_id}.mimiciv_3_1_icu.chartevents`
WHERE itemid IN ({', '.join(map(str, mobilization_related_itemids))})
"""
df_mobility = client.query(query).to_dataframe()
df_mobility.head()

,subject_id,hadm_id,stay_id,caregiver_id,charttime,storetime,itemid,value,valuenum,valueuom,warning
0,13668411,29071109,33246504,88074,2187-04-02 20:00:00,2187-04-02 20:45:00,227345,Weak,10.0,None,0
1,15009741,21688347,38650305,22746,2128-07-21 08:00:00,2128-07-21 15:09:00,227345,Weak,10.0,None,0
2,16945679,22590831,35624017,62592,2114-06-11 00:16:00,2114-06-11 00:16:00,227345,Weak,10.0,None,0
3,17951619,27934317,35128182,33363,2183-11-08 20:00:00,2183-11-08 22:44:00,227345,Weak,10.0,None,0
4,16528873,28073029,38261855,95684,2181-03-24 10:16:00,2181-03-24 10:16:00,227345,Weak,10.0,None,0


In [367]:
df_mobility['value'].unique()

array(['Weak', 'Normal ', 'Bed rest', 'Immobile', 'Impaired', 'Ambulate',
       'Both Arms', 'Both Legs', 'All Extremities', 'Active ', 'Passive',
       'Passive ROM', 'mod A', 'Not Applicable', 'Squat pivot',
       'Stand step', 'OOB', 'Active ROM', 'Contact guarding', 'min A',
       'Supervision', 'Independent', 'max A', 'Stand pivot',
       'Slide board'], dtype=object)

In [368]:
# Pre-defined list of itemids with comments as descriptions
mobilization_related_itemids_with_descriptions = {
    224092: "Range of Motion Location",
    225176: "Range of Motion Status",
    227345: "Gait/Transferring",
    227847: "Sit to Stand",
    227848: "Transfer",
    228697: "Mobilization Plan",
}

mobilization_dict = {}

# Iterate over each itemid in the prioritized list with descriptions
for itemid, description in mobilization_related_itemids_with_descriptions.items():
    # Filter the DataFrame for rows with the current itemid
    filtered_df = df_mobility[df_mobility['itemid'] == itemid]

    # Get the unique values for these rows and their value counts
    unique_values = filtered_df['value'].value_counts().to_dict()

    # Assign a dictionary with description, unique values, and their counts to the itemid in the final dictionary
    mobilization_dict[itemid] = {"data description": description, "data unique values": unique_values}

mobilization_dict

{224092: {'data description': 'Range of Motion Location',
  'data unique values': {'All Extremities': 146152,
   'Both Arms': 10407,
   'Both Legs': 2389,
   'Not Applicable': 365}},
 225176: {'data description': 'Range of Motion Status',
  'data unique values': {'Passive': 80939, 'Active ': 75563}},
 227345: {'data description': 'Gait/Transferring',
  'data unique values': {'Bed rest': 559991,
   'Weak': 92438,
   'Normal ': 62733,
   'Impaired': 54050,
   'Immobile': 13964}},
 227847: {'data description': 'Sit to Stand',
  'data unique values': {'min A': 49,
   'mod A': 44,
   'Contact guarding': 34,
   'max A': 33,
   'Supervision': 9,
   'Independent': 8}},
 227848: {'data description': 'Transfer',
  'data unique values': {'Stand step': 48,
   'min A': 32,
   'mod A': 30,
   'max A': 29,
   'Contact guarding': 20,
   'Stand pivot': 19,
   'Squat pivot': 9,
   'Supervision': 6,
   'Independent': 3,
   'Slide board': 1}},
 228697: {'data description': 'Mobilization Plan',
  'data uni

# Component F

In [369]:
regex= r"(?i)\bFamily\b"
regex_output= client.query(f"""
SELECT *
FROM `{dataset_id}.mimiciv_3_1_icu.d_items`
WHERE REGEXP_CONTAINS(
  LOWER(CONCAT(' ',
    IFNULL(label,''),' ',
    IFNULL(abbreviation,''),' '
  )),
  r\"\"\"{regex}\"\"\"
)
""").to_dataframe()
regex_output

,itemid,label,abbreviation,linksto,category,unitname,param_type,lownormalvalue,highnormalvalue
0,228125,Family meeting held,Family meeting held,procedureevents,7-Communication,None,Processes,NaN,NaN
1,228126,Family met with Case Manager,Family met with Case Manager,procedureevents,7-Communication,None,Processes,NaN,NaN
2,228127,Family met with Social Worker,Family met with Social Worker,procedureevents,7-Communication,None,Processes,NaN,NaN
3,228128,Family updated by MD,Family updated by MD,procedureevents,7-Communication,None,Processes,NaN,NaN
4,228129,Family updated by RN,Family updated by RN,procedureevents,7-Communication,None,Processes,NaN,NaN
5,228136,Family notified of transfer,Family notified of transfer,procedureevents,7-Communication,None,Processes,NaN,NaN
6,228228,"Family meeting attempted, unable","Family meeting attempted, unable",procedureevents,7-Communication,None,Processes,NaN,NaN
7,224024,Family Communication,Family Communication,chartevents,Restraint/Support Systems,None,Text,NaN,NaN
8,227662,Patient/Family Informed_V2,Patient/Family Informed_V2,chartevents,Restraint/Support Systems,None,Text,NaN,NaN
9,227960,Patient/Family Informed,Patient/Family Informed,chartevents,Restraint/Support Systems,None,Text,NaN,NaN


In [370]:
family_related_itemids =  [
    224024, # Family Communication
    224447, # Family Meeting
    224855, # Patient/Family Informed_V1
    227662, # Patient/Family Informed_V2
    227960, # Patient/Family Informed
    228264, # FM Condition Rev
    228276, # FM Prognosis reviewed
    228271, # FM Interpreter present
    228279, # FM Spirit/cultural
    228285  # FM Tx/goals
]
query = f"""
SELECT *
FROM `{dataset_id}.mimiciv_3_1_icu.chartevents`
WHERE itemid IN ({', '.join(map(str, family_related_itemids))})
"""
df_family = client.query(query).to_dataframe()
df_family.head()

,subject_id,hadm_id,stay_id,caregiver_id,charttime,storetime,itemid,value,valuenum,valueuom,warning
0,17104139,20201177,38041019,41632,2188-11-13 08:00:00,2188-11-13 10:57:00,224024,Family Called,NaN,None,0
1,15504355,26341600,30948346,46327,2128-04-30 08:30:00,2128-04-30 08:30:00,224024,Family Called,NaN,None,0
2,17436136,29932581,38848421,77930,2144-10-08 10:00:00,2144-10-08 17:46:00,224024,Family Called,NaN,None,0
3,15375637,22967708,36739309,94613,2159-11-27 10:19:00,2159-11-27 10:19:00,224024,Family Called,NaN,None,0
4,14871428,27449756,36269414,14453,2155-09-18 12:01:00,2155-09-18 12:02:00,224024,Family Called,NaN,None,0


In [371]:
df_family['value'].unique()

array(['Family Called', 'Family Visited', 'Family Conference',
       'Family Talked to MD', 'Family Talked to RN',
       'Social Service Involved', 'No', 'Yes', 'Not applicable', '-----',
       '1', '0'], dtype=object)

In [372]:
# Pre-defined list of itemids with comments as descriptions
family_related_itemids_with_description =  {
    224024:"Family Communication",
    224447: "Family Meeting",
    224855: "Patient/Family Informed_V1",
    227662: "Patient/Family Informed_V2",
    227960: "Patient/Family Informed",
    228264: "FM Condition Rev",
    228276: "FM Prognosis reviewed",
    228271: "FM Interpreter present",
    228279: "FM Spirit/cultural",
    228285: "FM Tx/goals"
}

family_dict = {}

# Iterate over each itemid in the prioritized list with descriptions
for itemid, description in family_related_itemids_with_description .items():
    # Filter the DataFrame for rows with the current itemid
    filtered_df = df_family[df_family['itemid'] == itemid]

    # Get the unique values for these rows and their value counts
    unique_values = filtered_df['value'].value_counts().to_dict()

    # Assign a dictionary with description, unique values, and their counts to the itemid in the final dictionary
    family_dict[itemid] = {"data description": description, "data unique values": unique_values}

family_dict

{224024: {'data description': 'Family Communication',
  'data unique values': {'Family Visited': 299565,
   'Family Talked to RN': 264368,
   'Family Called': 120098,
   'Family Talked to MD': 100100,
   'Social Service Involved': 11937,
   'Family Conference': 6245}},
 224447: {'data description': 'Family Meeting',
  'data unique values': {'1': 1167, '0': 681}},
 224855: {'data description': 'Patient/Family Informed_V1',
  'data unique values': {'1': 1637, '0': 89}},
 227662: {'data description': 'Patient/Family Informed_V2',
  'data unique values': {'Yes': 36002, 'No': 4382, 'Not applicable': 1528}},
 227960: {'data description': 'Patient/Family Informed',
  'data unique values': {'Yes': 812822, 'No': 139214, '-----': 115133}},
 228264: {'data description': 'FM Condition Rev', 'data unique values': {}},
 228276: {'data description': 'FM Prognosis reviewed',
  'data unique values': {}},
 228271: {'data description': 'FM Interpreter present',
  'data unique values': {}},
 228279: {'dat